In [4]:
import pandas as pd
import re
import os

# ============================================
# 1. COMBINE ALL MONTHLY FILES
# ============================================

files = {
    "July": "POWERSTAR JUL.xlsx",
    "August": "POWERSTAR AUG.xlsx",
    "September": "POWERSTAR SEP.xlsx",
    "October": "POWERSTAR OCT.xlsx",
    "November": "POWERSTAR NOV.xlsx"
}

df_list = []

for month, file in files.items():
    if os.path.exists(file):
        temp = pd.read_excel(file, sheet_name="DATA")
        temp["Month"] = month
        df_list.append(temp)
        print(f"Loaded {file}")
    else:
        print(f"❌ Missing file: {file}")

df = pd.concat(df_list, ignore_index=True)
print("\n✅ All files combined!")
# Standardize suppliers by using only the first word
df['Supplier_Std'] = (
    df['Supplier']
    .str.strip()
    .str.lower()
    .str.split()
    .str[0]
)

# ============================================
# 2. STANDARDIZE DEPARTMENT (FIRST SCRIPT)
# ============================================

def standardize_department(dept):
    if pd.isna(dept):
        return None

    dept = str(dept).lower().strip()
    dept = re.sub(r'[^a-z0-9\s]', ' ', dept)
    dept = re.sub(r'\s+', ' ', dept)

    if 'bleach' in dept:
        return 'Bleach'
    elif any(x in dept for x in ['light', 'candle']):
        return 'Candles'
    elif any(x in dept for x in [
        'dishwash', 'dishwahing', 'soaps dishwashing', 'dish washing',
        'd/wash', 'dwl', 'd/l'
    ]):
        return 'Dishwashing'
    elif 'fabric soften' in dept:
        return 'Fabric Softeners'
    elif any(x in dept for x in ['h/w', 'hand wash', 'handwash', 'h/wash', 'hand/w']):
        return 'Handwash'
    elif any(x in dept for x in ['ball', 'balls']):
        return 'Toilet Balls'
    elif any(x in dept for x in ['block', 'blocks']):
        return 'Air Freshener Blocks'
    elif any(x in dept for x in ['scour', 'scouring', 'scoring', 'scouring powder', 'scour powder']):
        return 'Scouring Powders'
    elif any(x in dept for x in [
        'toilet cleaner', 'toiletcleaner', 'toilet gel', 'toilet liquid',
        'toilet liquids', 'toilet cleaner liquids', 'toilet cleanerliquid', 'toilet clener liquids'
    ]):
        return 'Toilet Cleaners'
    elif 'shower gel' in dept:
        return 'Body Wash'
    elif 'mineral water' in dept:
        return 'Water'
    elif 'water' in dept:
        return 'Water Purifier'
    elif 'air' in dept:
        return 'Air Fresheners'
    else:
        return dept.title()

df['Category'] = df['Department'].apply(standardize_department)

print("✅ Department standardization done!")

# ============================================
# 3. CATEGORY FIXES BASED ON DESCRIPTION
# ============================================
# Standardize supplier names by taking first word (lowercase to remove variation)

df['Description'] = df['Description'].astype(str).str.lower().str.strip()
df['Category'] = df['Category'].astype(str).str.strip()

def fix_category(row):
    desc = row['Description']
    cat = row['Category']

    # ---------------------------------------------------------
    # WINDOW CLEANERS vs GLASS CLEANERS (NEW LOGIC)
    # ---------------------------------------------------------
    window_glass_keywords = [
        'magn glass cleaner','safish wdw cleaner','jet window cleaner',
        'topex lemon w/cleaner','topex lavender w/cleaner',
        'topex window cleaner','safisha glass','glass cleaner',
        'glasscln','glass clnr','glass cln','velvexwindow',
        'tintewindow','g/cleaner','g.cleaner',"velvex w/cleaner" 'v.w/cleaner'
    ]

    if any(x in desc for x in window_glass_keywords):

        # Sizes that convert to WINDOW CLEANER
        if '300ml' in desc or '320ml' in desc:
            return 'Window Cleaner'

        # Otherwise it is a GLASS CLEANER
        return 'Glass Cleaner'

    # ---------------------------------------------------------
    # MULTIPURPOSE
    # ---------------------------------------------------------
    if any(x in desc for x in [
            'magnee pine', 'dry cleaner', 'tile', 'm/puro',
            'magnee cleaner','magnee lavender', 'magnee multi','claymulti']):
            return 'All Purpose'
    if any(x in desc for x in [
        'multipurpose', 'multi', 'multip', 'multp', 'multi/p',
        'multi-purp', 'm/purpose', 'multi-kleen', 'multikleen',
        'multkleen', 'multi kleen','mult-purp'
    ]):
        return 'Multipurpose'
    
    if 'block' in desc or 'blocks' in desc:
        return 'Air Freshener Blocks'

    # ---------------------------------------------------------
    # BLEACH SPLITTING
    # ---------------------------------------------------------
    if any(x in desc for x in ['disin', 'disinfectant', 'desinfectant','magnee pine', 'desnifectant']):
            return 'Disinfectant'
    if 'bleach' in cat.lower() or 'bleach' in desc:
        if any(x in desc for x in ['colour', 'color', 'colors', 'colours']):
            return 'Bleach - Colours'
        elif any(x in desc for x in ['disin', 'disinfectant', 'desinfectant']):
            return 'Disinfectant'
        elif any(x in desc for x in [
            'magnee pine', 'dry cleaner', 'tile', 'm/puro',
            'magnee cleaner','magnee lavender', 'magnee multi','claymulti']):
            return 'All Purpose'
        elif any(x in desc for x in [
            'multipurpose', 'multi p', 'multip', 'multp', 'multi/p', 'm/purpose', 'multi-kleen', 'multikleen',
            'multkleen', 'multi kleen','mult-purp'
        ]):
            return 'Multipurpose'
        
        else:
            return 'Bleach - Regular'

    # ---------------------------------------------------------
    # HANDWASH
    # ---------------------------------------------------------
    if any(x in desc for x in ['handwash', 'hand wash', 'h/wash', 'h wash', 'handwsh']) and \
       not any(x in desc for x in ['dish wash', 'dish washing', 'dish-wash', 'dish/w']):
        return 'Handwash'

    # ---------------------------------------------------------
    # BODY WASH
    # ---------------------------------------------------------
    if any(x in desc for x in ['body wash', 'b/wash', 'b wash', 'shower gel', 'bdwash']):
        return 'Body Wash'
    if any (x in desc for x in ['aqua guard', 'aquarguard', 'waterguard', 'water guard']):
        return "Water Purifier"

    # ---------------------------------------------------------
    # ALL PURPOSE CLEANERS
    # ---------------------------------------------------------
    if any(x in desc for x in [
        'magnee lavender', 'a/purpose', 'c/uphostery', 'tile',
        'ultraclean lav disinfectant', 'mult'
    ]):
        return 'All Purpose'

    # ---------------------------------------------------------
    # DISHWASHING
    # ---------------------------------------------------------
    if any(x in desc for x in [
        'dw', 'dwscouring', 'dishwashing', 'dish w', 'd/wash', 'dishwash',
        'dish wash', 'dish washing', 'dish/w'
    ]) and not any(x in desc for x in [
        'handwash', 'hand wash', 'h/wash', 'body wash', 'b/wash', 'b wash',
        'bleach', 'tintewindow'
    ]):
        return 'Dishwashing'

    # ---------------------------------------------------------
    # SCOURING
    # ---------------------------------------------------------
    if any(x in desc for x in [
        'scouring powder', 'toiletcleaner scouring powder',
        'safisha scouring', 'safisha s/powder', 'scourer',"scouring",'scour'
    ]):
        return 'Scouring Powders'

    # ---------------------------------------------------------
    # MAGNEE FIX
    # ---------------------------------------------------------
    if 'magnee' in desc and cat.lower() == 'toilet cleaners':
        return 'All Purpose'

    # ---------------------------------------------------------
    # BALLS & BLOCKS
    # ---------------------------------------------------------
    if 'ball' in desc or 'balls' in desc:
        return 'Toilet Balls'



    # ---------------------------------------------------------
    # GLUE
    # ---------------------------------------------------------
    if 'glue' in desc:
        return "Office Glue"
    if 'ariel' in desc:
        return "Detergent"

    return cat


df['Category'] = df.apply(fix_category, axis=1)

print("✅ Category cleanup based on Description complete!")

# ============================================
# 4. SAVE FINAL CLEANED FILE
# ============================================

df.to_excel("POWERSTAR_FINAL_CLEANED.xlsx", index=False)

print("\n🎉 ALL DONE! File saved as POWERSTAR_FINAL_CLEANED.xlsx")
print(df['Category'].value_counts())


Loaded POWERSTAR JUL.xlsx
Loaded POWERSTAR AUG.xlsx
Loaded POWERSTAR SEP.xlsx
Loaded POWERSTAR OCT.xlsx
Loaded POWERSTAR NOV.xlsx

✅ All files combined!
✅ Department standardization done!
✅ Category cleanup based on Description complete!

🎉 ALL DONE! File saved as POWERSTAR_FINAL_CLEANED.xlsx
Category
Stationery Pens Rulers Etc     4688
Toilet Cleaners                3698
Fabric Softeners               2965
Handwash                       2861
Stationary Others              2646
Stationery Pens Rulers         2595
Bleach - Regular               2423
Dishwashing                    2319
Air Fresheners                 1797
Scouring Powders               1224
Bleach - Colours                850
Water                           828
Body Wash                       824
Toilet Balls                    774
Office Glue                     762
Candles                         753
Air Freshener Blocks            493
Stationery Others               418
Multipurpose                    236
Water Purifie

In [5]:
import pandas as pd

# Load your combined file
df = pd.read_excel("POWERSTAR_FINAL_CLEANED.xlsx")

# Create a mapping dictionary for handlers
handler_map = {
    "Express": "Isabella Wambui",
    "Hyper": "N/A",
    "Jambo": "Stephen Otieno",
    "Kangari": "Margaret Njeri",
    "Kasarani": "Linet Atieno",
    "Kikuyu": "Dorcas Paul",
    "Kinoo": "Dorcas Paul",
    "Kitengela": "Peris Wavinya",
    "Mini": "Stephen Otieno",
    "Vasha": "Francis Murage",
    "Zimmerman": "Isabella Wambui"
}

# Create a new column 'Handler'
df["Handler"] = df["Branch"].map(handler_map)

# Save updated file
df.to_excel("POWERSTAR_HANDLERS_ADDED.xlsx", index=False)


In [6]:
import pandas as pd

# Load the file from the previous step
df = pd.read_excel("POWERSTAR_HANDLERS_ADDED.xlsx")

# Create region mapping (example — update to your real regions)
region_map = {
    "Express": "Nairobi",
    "Hyper": "Nairobi",
    "Jambo": "Murang’a",
    "Kangari": "Murang’a",
    "Kasarani": "Nairobi",
    "Kikuyu": "Kiambu",
    "Kinoo": "Kiambu",
    "Kitengela": "Kajiado",
    "Mini": "Murang’a",
    "Vasha": "Nakuru",
    "Zimmerman": "Nairobi"
}

# Create a new column 'Region'
df["Region"] = df["Branch"].map(region_map)

# Save final file
df.to_excel("POWERSTAR_COMBINED_DATA.xlsx", index=False) 
